In [26]:
import numpy as np
import matplotlib.pyplot as plt

Name: Pathri Vidya Praveen <br>
Roll number: CS24BTECH11047 <br>
References/resources used to write this program: Lecture Notes

In [27]:
def integer_prefixencode(a: int) -> str:
    '''
    Return the prefix code for a given integer a.
    The input is a positive integer a.
    The output should be a 0-1 string representing the prefix code for a (as discussed in class).
    '''
    binary_a = bin(a)[2:]
    length = len(binary_a)
    length_binary = bin(length)[2:]
    unary_prefix = "0"*(len(length_binary)-1) + "1"
    codeword = unary_prefix + length_binary + binary_a

    return(codeword)

def integer_prefixdecode(codeword: str) -> int:
    '''
    Return the integer corresponding to a given prefix code.
    The input is a 0-1 string representing the prefix code for some integer a (as discussed in class).
    The output should be the positive integer a corresponding to the input codeword.
    '''
    i= 0
    while codeword[i] == "0":
      i = i+1
    i = i+1
    length_l = i
    l = int(codeword[i:i+length_l],2)
    i = i + length_l
    a = int(codeword[i:i+l],2)

    return(a)

In [29]:
def lz77_encode(s: str, alphabet: list) -> tuple[list, str]:
    '''
    Return the LZ77 encoding and the corresponding binary representation of a given string s.
    The input is a string s.
    The first part of the output should be a list of tuples (flag, newchar) or (flag, pointer, length) representing the LZ77 encoding of s.
    The binary representation should be a 0-1 string.
    '''

    n = len(s)
    t = 0
    lz77_list = []
    lz77_binary = ""
    b = (len(alphabet) - 1).bit_length()
    char_to_index = {c: i for i,c in enumerate(alphabet)}
    while t<n:
      max_length = 0
      max_pos = 0
      for j in range(t):
        length = 0
        while t < n - length and s[j+length] == s[t+length]:
          length = length + 1

        if length > max_length:
          max_length = length
          max_pos = j

      if max_length > 0:
        lz77_list.append((1,t-max_pos,max_length))
        lz77_binary += "1"
        lz77_binary += integer_prefixencode(t-max_pos)
        lz77_binary += integer_prefixencode(max_length)
        t = t + max_length

      else:
        lz77_list.append((0,s[t]))
        lz77_binary += "0"
        if b != 0:
          idx = char_to_index[s[t]]
          lz77_binary += format(idx,f'0{b}b')
        t = t + 1


    return(lz77_list, lz77_binary)


def lz77_decode(lz77_binary: str, alphabet: list) -> str:
    '''
    Return the decoded string from a given LZ77 binary representation.
    The input is a 0-1 string representing the LZ77 encoding of a string.
    The output should be the original string that was encoded.
    '''
    i = 0
    decoded_string = ""
    b = (len(alphabet) -1).bit_length()
    while i < len(lz77_binary):
      if lz77_binary[i] == "0":
        i = i+1
        if b == 0:
          decoded_string += alphabet[0]
        else:
          idx = int(lz77_binary[i:i+b], 2)
          decoded_string += alphabet[idx]
          i = i+b

      else:
        i = i+1
        start = i
        while lz77_binary[i] == "0":
          i = i+1
        i = i+1
        length_l = i-start
        length_bits = int(lz77_binary[i:i+length_l],2)
        i = i+length_l
        p = int(lz77_binary[i:i+length_bits],2)
        i = i+length_bits

        start = i
        while lz77_binary[i] == "0":
          i = i+1
        i =i+1
        length_l = i-start
        length_bits = int(lz77_binary[i:i+length_l],2)
        i = i+length_l
        l = int(lz77_binary[i:i+length_bits],2)
        i = i + length_bits

        start_pos = len(decoded_string)-p
        for k in range(l):
          decoded_string += decoded_string[start_pos+k]

    return(decoded_string)


In [30]:
def lz78_encode(s: str, alphabet: list) -> tuple[list, str]:
    '''
    Return the LZ78 encoding and the corresponding binary representation of a given string s.
    The input is a string s.
    The first part of the output should be a list of tuples (index, newchar) representing the LZ78 encoding of s.
    The binary representation should be a 0-1 string.
    '''
    dictionary = {}
    current = ""
    index = 1
    lz78_list = []
    lz78_binary = ""
    b = max(1, (len(alphabet) - 1).bit_length())
    char_to_index = {c: i for i,c in enumerate(alphabet)}

    for c in s:
      if current + c in dictionary:
        current = current + c
      else:
        if current == "":
          lz78_list.append((0,c))
          lz78_binary += integer_prefixencode(0)
          if b != 0:
            idx = char_to_index[c]
            lz78_binary += format(idx, f'0{b}b')
        else:
          lz78_list.append((dictionary[current],c))
          lz78_binary += integer_prefixencode(dictionary[current])
          if b != 0:
            idx = char_to_index[c]
            lz78_binary += format(idx, f'0{b}b')

        dictionary[current+c] = index
        index = index + 1
        current = ""

    if current != "":
      lz78_list.append((dictionary[current],""))
      lz78_binary += integer_prefixencode(dictionary[current])

    return(lz78_list, lz78_binary)

def lz78_decode(lz78_binary: str, alphabet: list) -> str:
    '''
    Return the decoded string from a given LZ78 binary representation.
    The input is a 0-1 string representing the LZ78 encoding of a string.
    The output should be the original string that was encoded.
    '''
    i = 0
    dictionary = {0: ""}
    index = 1
    decoded_string = ""
    b = max(1, (len(alphabet) - 1).bit_length())

    while i < len(lz78_binary):
      start = i
      while lz78_binary[i] == "0":
        i = i+1

      i=i+1
      length_l = i-start
      length_bits = int(lz78_binary[i:i+length_l],2)
      i = i+length_l
      idx = int(lz78_binary[i:i+length_bits],2)
      i = i+length_bits

      if b == 0:
        if i < len(lz78_binary):
          char = alphabet[0]
        else:
          char = ""
      else:
        if i + b <= len(lz78_binary):
          char_index = int(lz78_binary[i:i+b], 2)
          char = alphabet[char_index]
          i = i+b
        else:
          char = ""

      entry = dictionary[idx] + char
      decoded_string += entry
      dictionary[index] = entry
      index = index + 1

    return(decoded_string)




In [31]:
# Your code must pass this, and other tests. Do not change this part of the code in your submission
test_strings = ['AABCAA', 'ABABAB']
compressed_sizes_lz77 = []
compressed_sizes_lz78 = []

for s in test_strings:
    alphabet = list(set(s))
    # print(f"Testing LZ77 encoding and decoding for string: {s}")
    lz77_list, lz77_binary = lz77_encode(s, alphabet)
    # print(f"LZ77 Encoding (list): {lz77_list}")
    # print(f"LZ77 Encoding (binary): {lz77_binary}")
    decoded_string = lz77_decode(lz77_binary, alphabet)
    # print(f"Decoded string: {decoded_string}")
    compressed_sizes_lz77.append(len(lz77_binary))
    if decoded_string != s:
        print("LZ77 decoding failed for string:", s)

    # print(f"Testing LZ78 encoding and decoding for string: {s}")
    lz78_list, lz78_binary = lz78_encode(s, alphabet)
    # print(f"LZ78 Encoding (list): {lz78_list}")
    # print(f"LZ78 Encoding (binary): {lz78_binary}")
    decoded_string = lz78_decode(lz78_binary, alphabet)
    # print(f"Decoded string: {decoded_string}")
    compressed_sizes_lz78.append(len(lz78_binary))
    if decoded_string != s:
        print("LZ78 decoding failed for string:", s)

print("Compressed sizes for LZ77:", compressed_sizes_lz77)
print("Compressed sizes for LZ78:", compressed_sizes_lz78)

Compressed sizes for LZ77: [30, 18]
Compressed sizes for LZ78: [20, 18]
